# 🎬 YouTube AI Video Studio — Micro-Clip Frame-Sync Engine
Run a complete **Micro-Clip Audio-Visual Frame-Locked Studio** on Google's free T4 GPU!

In [ ]:
# @title 1. Install Required Packages
!pip install -q gradio edge-tts diffusers transformers accelerate torch torchvision torchaudio pillow imageio-ffmpeg


In [ ]:
import asyncio, os, re, subprocess, traceback, torch
import gradio as gr
import edge_tts
from diffusers import AutoPipelineForText2Image
from PIL import Image

print('⚡ Loading GPU Stable Diffusion Engine...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
pipe = AutoPipelineForText2Image.from_pretrained('stabilityai/sdxl-turbo', torch_dtype=torch.float16, variant='fp16')
pipe.to(device)
print(f'✅ Stable Diffusion loaded on {device}!')

VOICE_MAP = {
    'Andrew (YouTube Documentary)': 'en-US-AndrewNeural',
    'Christopher (Deep Storyteller)': 'en-US-ChristopherNeural',
    'Ava (Modern Expressive)': 'en-US-AvaNeural',
    'Guy (News & Commentary)': 'en-US-GuyNeural'
}

STYLE_PREFIXES = {
    '2D Cartoon / Explainer': (
        '100% 2D hand-drawn editorial economics cartoon illustration, professional educational cartoon style, whiteboard-inspired artwork, '
        'thick slightly imperfect black ink outlines, sketchy marker strokes, subtle paper grain, 60-30-10 color harmony with warm off-white canvas. '
        'STRICT NO PHOTOGRAPHY RULE: ABSOLUTELY NO REALISTIC PHOTOGRAPHY, NO REAL HUMAN PHOTOS, NO REALISTIC PEOPLE OR PHOTO-REALISTIC BACKGROUNDS. '
        'ALL BACKGROUND PEOPLE AND ENVIRONMENTS MUST BE 2D HAND-DRAWN CARTOON FIGURES. '
    ),
    'Photorealistic 8K': 'Hyperrealistic 8K documentary photography, cinematic lighting, 35mm lens, depth of field, detailed. ',
    'Cinematic Movie': 'Dramatic cinematic movie scene, anamorphic lens, 8k resolution, film grain, movie still. ',
    'Studio Anime': 'High quality studio anime aesthetic, crisp lines, vibrant colors, detailed background art. ',
    'Neon Cyberpunk': 'Futuristic neon cyberpunk city aesthetic, glowing cyan and magenta lights, rainy moody night. '
}

ZENN_CHARACTER_SNIPPET = (
    'Featuring the central recurring character: a simple hand-drawn expressive 2D stick figure guide with clean black ink outlines, '
    'wearing a vibrant crimson-red backwards baseball cap, an eye-catching electric-blue oversized hoodie, deep indigo baggy jeans, '
    'fresh white sneakers, and a prominent giant glowing metallic gold dollar-sign ($) medallion necklace, standing out as the colorful narrator. '
    'STRICT NO LABELS RULE: DO NOT WRITE THE WORDS "HOST", "HOST 3", OR ANY POINTER ARROWS ON OR NEAR THE CHARACTER. '
)

ZENN_SUFFIX = (
    'Showing a grand 2D hand-drawn physical environment. '
    'ABSOLUTELY NO INFOGRAPHIC SLIDES, NO TOP CATEGORY HEADINGS, NO SPLIT-SCREEN DIAGRAM BOXES, AND NO CONNECTING ARROWS. '
    'Professional YouTube economics explainer documentary aesthetic.'
)

CURRENT_STATE = {'beats': []}

def parse_input_script_or_cards(raw_input, style_name):
    beats = []
    raw_input_str = raw_input.strip()
    if '### SCRIPT LINE:' in raw_input_str or '### IMAGE PROMPT:' in raw_input_str:
        card_blocks = re.split(r'(?=# Beat|### SCRIPT LINE:)', raw_input_str)
        for block in card_blocks:
            if not block.strip(): continue
            line_match = re.search(r'### SCRIPT LINE:\s*["\']?(.*?)["\']?\s*(?=###|#|$)', block, re.DOTALL)
            prompt_match = re.search(r'### IMAGE PROMPT:\s*(.*?)\s*(?=###|#|$)', block, re.DOTALL)
            if line_match:
                script_line = line_match.group(1).strip().strip('"')
                custom_prompt = prompt_match.group(1).strip() if prompt_match else build_zenn_image_prompt(script_line, style_name)
                beats.append({'text': script_line, 'prompt': custom_prompt})
    else:
        lines = [l.strip() for l in raw_input_str.split('\n') if l.strip()]
        for line in lines:
            chunks = [c.strip() for c in re.split(r'(?<=[,;:!?.—])\s+', line) if c.strip()]
            for chunk in chunks:
                words = chunk.split()
                if len(words) > 10:
                    sub_chunks = [sc.strip() for sc in re.split(r'(?<=[,])\s+', chunk) if sc.strip()]
                    for sc in sub_chunks: beats.append({'text': sc, 'prompt': build_zenn_image_prompt(sc, style_name)})
                else: beats.append({'text': chunk, 'prompt': build_zenn_image_prompt(chunk, style_name)})
    return beats if beats else [{'text': raw_input_str, 'prompt': build_zenn_image_prompt(raw_input_str, style_name)}]

def build_zenn_image_prompt(beat_text, style_name):
    prefix = STYLE_PREFIXES.get(style_name, STYLE_PREFIXES['2D Cartoon / Explainer'])
    clean_line = re.sub(r'\s+', ' ', beat_text).strip()
    money_match = re.search(r'(\$?\d+[\d,.]*\s*(million|billion|thousand|k|m)?)', clean_line, re.IGNORECASE)
    money_callout = f' Hand-drawn financial text label showing "{money_match.group(0).upper()}".' if (money_match and len(money_match.group(0))>1) else ''
    narrator_keywords = ['you', 'your', 'we', 'our', 'welcome', 'let\'s', 'okay', 'so', 'look', 'here', 'problem', 'except']
    has_narrator = any(re.search(rf'\b{kw}\b', clean_line, re.IGNORECASE) for kw in narrator_keywords)
    char_str = ZENN_CHARACTER_SNIPPET if (has_narrator and '2D Cartoon' in style_name) else ''
    return f'{prefix} {char_str}A single full-frame 16:9 2D cartoon physical location scene depicting: "{clean_line}". {money_callout} {ZENN_SUFFIX}'

def get_media_duration(file_path):
    cmd = ['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    try: return float(res.stdout.strip())
    except: return 1.5

async def generate_micro_scene_audio(text, voice_id, raw_audio_path, padded_audio_path):
    communicate = edge_tts.Communicate(text, voice_id, rate='+0%', pitch='+0Hz')
    await communicate.save(raw_audio_path)
    af_filters = 'highpass=f=75,equalizer=f=120:t=q:w=1:g=2.5,compand=attacks=0.03:decays=0.3:points=-60/-60|-24/-14|-12/-8|0/-3,apad=pad_dur=0.30'
    cmd = ['ffmpeg', '-y', '-i', raw_audio_path, '-af', af_filters, '-c:a', 'libmp3lame', '-b:a', '192k', padded_audio_path]
    subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    return padded_audio_path if os.path.exists(padded_audio_path) else raw_audio_path

def create_micro_clip_engine(script_input, voice_name, style_name, format_name):
    try:
        if not script_input.strip(): return 'Please enter a script or beat cards.', None, [], None
        voice_id = VOICE_MAP.get(voice_name, 'en-US-AndrewNeural')
        width, height = (1280, 720) if '16:9' in format_name else (720, 1280)
        sd_width, sd_height = (1024, 576) if '16:9' in format_name else (576, 1024)
        os.makedirs('micro_clips', exist_ok=True); os.makedirs('micro_audio', exist_ok=True); os.makedirs('micro_images', exist_ok=True)
        beats = parse_input_script_or_cards(script_input, style_name)
        try: loop = asyncio.get_event_loop()
        except RuntimeError: loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        generated_images, clip_files = [], []
        status_log = [f'🎬 Processing {len(beats)} Micro-Clips with 300ms Natural Human Pauses...\n']
        for idx, beat in enumerate(beats, start=1):
            raw_audio, padded_audio = f'micro_audio/scene_{idx}_raw.mp3', f'micro_audio/scene_{idx}_padded.mp3'
            img_path, clip_path = f'micro_images/scene_{idx}.jpg', f'micro_clips/scene_{idx}_clip.mp4'
            loop.run_until_complete(generate_micro_scene_audio(beat['text'], voice_id, raw_audio, padded_audio))
            exact_dur = get_media_duration(padded_audio)
            img = pipe(beat['prompt'], num_inference_steps=3, guidance_scale=0.0, width=sd_width, height=sd_height).images[0]
            img.save(img_path); generated_images.append(img_path)
            cmd_clip = ['ffmpeg', '-y', '-loop', '1', '-i', os.path.abspath(img_path), '-i', os.path.abspath(padded_audio), '-vf', f'scale={width}:{height}:force_original_aspect_ratio=decrease,pad={width}:{height}:(ow-iw)/2:(oh-ih)/2:color=black', '-c:v', 'libx264', '-tune', 'stillimage', '-pix_fmt', 'yuv420p', '-c:a', 'aac', '-b:a', '192k', '-shortest', clip_path]
            res_clip = subprocess.run(cmd_clip, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if res_clip.returncode == 0 and os.path.exists(clip_path):
                clip_files.append(clip_path); status_log.append(f'✅ Scene {idx}/{len(beats)} [{exact_dur:.2f}s]: "{beat["text"]}"')
        concat_lines = [f'file \'{os.path.abspath(c)}\'' for c in clip_files]
        with open('concat_micro_clips.txt', 'w') as f: f.write('\n'.join(concat_lines))
        out_mp4 = 'final_youtube_video.mp4'
        cmd_concat = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', 'concat_micro_clips.txt', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-c:a', 'aac', out_mp4]
        subprocess.run(cmd_concat, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return '\n'.join(status_log) + f'\n\n🎉 Rendered {len(clip_files)} Frame-Locked Micro-Clips!', None, generated_images, out_mp4
    except Exception as e: return f'❌ Error: {str(e)}\n\n{traceback.format_exc()}', None, [], None

with gr.Blocks(title='YouTube AI Video Studio — Micro Clip Engine') as demo:
    gr.Markdown('# 🎬 YouTube AI Video Studio — Micro-Clip Frame-Sync Engine')
    gr.Markdown('Input raw scripts OR pre-formatted Beat Cards. Every scene gets its own parallel audio-visual mini clip + 300ms natural human pause!')
    with gr.Row():
        with gr.Column():
            script_input = gr.Textbox(label='YouTube Script or Beat Cards', lines=8)
            voice_dropdown = gr.Dropdown(choices=list(VOICE_MAP.keys()), value='Andrew (YouTube Documentary)', label='Narrator Voice')
            style_dropdown = gr.Dropdown(choices=list(STYLE_PREFIXES.keys()), value='2D Cartoon / Explainer', label='Visual Art Style')
            format_dropdown = gr.Dropdown(choices=['16:9 Widescreen (YouTube)', '9:16 Vertical (Shorts)'], value='16:9 Widescreen (YouTube)', label='Aspect Ratio')
            btn_auto = gr.Button('✨ 1-Click Auto-Generate Frame-Locked Video', variant='primary')
        with gr.Column():
            status_output = gr.Textbox(label='Status & Log Output', lines=8)
            audio_output = gr.Audio(label='Sample Audio Preview')
            gallery_output = gr.Gallery(label='Stable Diffusion Scene Images', columns=2)
            video_output = gr.Video(label='Final YouTube MP4 Video')
    btn_auto.click(create_micro_clip_engine, inputs=[script_input, voice_dropdown, style_dropdown, format_dropdown], outputs=[status_output, audio_output, gallery_output, video_output])
demo.launch(share=True, debug=True)
